In [24]:
## RUN this BEFORE job submission

import os

_new_directory = "/home/pkwon2/arm_maker" ## change to correct directory

print(f"The current working directory is: {os.getcwd()}")

os.chdir(_new_directory)
print(f"Changed working directory to: {os.getcwd()}")

def create_directory_structure(base_directory):
    try:
        os.makedirs(os.path.join(base_directory, 'input'), exist_ok=True)
        os.makedirs(os.path.join(base_directory, 'outputs'), exist_ok=True)
        print(f"Directory structure '{base_directory}/input' and '{base_directory}/output' created successfully.")
    except Exception as e:
        print(f"Error creating directory structure '{base_directory}': {e}")

if __name__ == "__main__":
    base_directory = "1_diffusion"
    create_directory_structure(base_directory)

The current working directory is: /home/pkwon2/arm_maker
Changed working directory to: /home/pkwon2/arm_maker
Directory structure '1_diffusion/input' and '1_diffusion/output' created successfully.


In [25]:
## Using DJ's branch

import itertools
import os

# Define the base command with placeholders
base_command = (
    "/software/containers/SE3nv.sif {script} "
    "--config-name=aa "
    "inference.ckpt_path='{ckpt_path}' "
    "inference.model_runner='NRBStyleSelfCond' "
    "inference.num_designs={num_designs} "
    "inference.output_prefix='{output_prefix}' "
    'contigmap.contigs="{contigs}" '
    "inference.two_template=True "
    "inference.three_template=True "
    "inference.rigid_repeat_motif=False "
    "diffuser.T=50 "
    "model.symmetrize_repeats=False "
    "model.symmsub_k=1 "
    "model.main_block=1 "
    "model.sym_method='max' "
    "denoiser.noise_scale_frame=0.05 "
    "denoiser.noise_scale_ca=0.05 "
    "inference.ij_visible='{ij_visible}' "
    "inference.motif_only_2d=True "
    "preprocess.eye_frames=True "
    "inference.supply_motif_seq=True "
    "inference.cyclic_protein_indices=True "
    "inference.input_pdb='{input_pdb}' "
    "inference.pseudo_symmetry='c{n_repeat}' "
    "model.pseudo_cycle=False "
    "inference.align_px0_motif=False"
)

# Script and checkpoint path variables
script = "/projects/small-molecule/rf_diffusion_pseudocycle/rf_diffusion/run_inference.py"
ckpt_paths = [
    "/projects/small-molecule/3template_pseudocycle/BFF_0.20.pt",
]

# Number of designs
num_des = 5  # Adjusted to match your example

# Define the output directory and ensure it exists
output_dir = "./1_diffusion/"
os.makedirs(output_dir, exist_ok=True)

# Define the output file path
tasks_file = os.path.join(output_dir, "tasks")

# Define the pdb_name variable
# pdb_1_name = 'combined_fd31_r6_pk_model_5_ptm_seed_0_unrelaxed_y22_C4_y-65_C2_LHD101_model_5_ptm_seed_0_unrelaxed_tile_a'
# pdb_2_name = 'combined_fd31_r6_pk_model_5_ptm_seed_0_unrelaxed_y22_C4_y-65_C2_LHD101_model_5_ptm_seed_0_unrelaxed_tile_b'
# pdb_3_name = 'c3'
# pdb_4_name = 'c4'
pdb_5_name = 'c2_tile_b'


# Example input dictionary with ranges for value1 and value2
input_pdb_dict = {
#     pdb_1_name: {
#         # Define ranges for value1 and value2
#         'contig_values': [list(range(98,118))],
#         'contigs_format': "['A77-130,23,A154-182,{value1},M38-75 A208-259']",
#         'num_designs': num_des,
#         'ij_visible': 'abcd',
#         'input_pdb': f'./input/{pdb_1_name}.pdb',
#         'n_repeat': '1',
#         'output_prefix': './outputs/',  # Output directory
#     },
#     pdb_2_name: {
#         # Define ranges for value1 and value2
#         'contig_values': [list(range(98, 118))],
#         'contigs_format': "['E77-130,23,E154-182,{value1},N112-150 E208-259']",
#         'num_designs': num_des,
#         'ij_visible': 'abcd',
#         'input_pdb': f'./input/{pdb_2_name}.pdb',
#         'n_repeat': '1',
#         'output_prefix': './outputs/',  # Output directory
#     },
    # pdb_3_name: {
    #     # Define ranges for value1 and value2
    #     'contig_values': [list(range(150,160))],
    #     'contigs_format': "['A208-259,{value1},C51-103']",
    #     'num_designs': num_des,
    #     'ij_visible': 'ab',
    #     'input_pdb': f'./input/{pdb_3_name}.pdb',
    #     'n_repeat': '1',
    #     'output_prefix': './outputs/',  # Output directory
    # },
    # pdb_4_name: {
    #     # Define ranges for value1 and value2
    #     'contig_values': [list(range(130, 140))],
    #     'contigs_format': "['A208-259,{value1},C51-103']",
    #     'num_designs': num_des,
    #     'ij_visible': 'ab',
    #     'input_pdb': f'./input/{pdb_4_name}.pdb',
    #     'n_repeat': '1',
    #     'output_prefix': './outputs/',  # Output directory
    # },
    pdb_5_name: {
        # Define ranges for value1 and value2
        'contig_values': [list(range(88, 108))],
        'contigs_format': "['I1-118,{value1},G39-75']",
        'num_designs': num_des,
        'ij_visible': 'ab',
        'input_pdb': f'./input/{pdb_5_name}.pdb',
        'n_repeat': '1',
        'output_prefix': './outputs/',  # Output directory
    },
}

# Function to generate all combinations of values from the provided lists
def generate_combinations(contig_lists):
    return list(itertools.product(*contig_lists))

# Write commands to the file
with open(tasks_file, "w") as file:
    for input_pdb_key, config in input_pdb_dict.items():
        contig_lists = config['contig_values']
        contigs_format = config['contigs_format']
        num_designs = config['num_designs']
        ij_visible = config['ij_visible']
        input_pdb = config['input_pdb']
        n_repeat = config.get('n_repeat', '')
        output_prefix_base = config['output_prefix']
        combinations = generate_combinations(contig_lists)

        for ckpt_path in ckpt_paths:
            for values in combinations:
                # Create a dictionary to hold value placeholders
                value_dict = {f"value{i+1}": str(value) for i, value in enumerate(values)}
                formatted_contigs = contigs_format.format(**value_dict)

                # Create a suffix based on the input PDB, checkpoint, and values
                suffix = "_".join([f"{k}_{v}" for k, v in value_dict.items()])
                suffix = f"{suffix}_ckpt_{os.path.basename(ckpt_path).split('.')[0]}"
                output_prefix_full = os.path.join(output_prefix_base, f"{input_pdb_key}_{suffix}")

                command = base_command.format(
                    script=script,
                    ckpt_path=ckpt_path,
                    num_designs=num_designs,
                    output_prefix=output_prefix_full,
                    contigs=formatted_contigs,
                    ij_visible=ij_visible,
                    input_pdb=input_pdb,
                    n_repeat=n_repeat,
                )

                file.write(command + "\n")

In [26]:
# Define the SBATCH header, change the email to your email so it doesn't ping me!
sbatch_header = """#!/bin/bash
#SBATCH -J diffusion
#SBATCH -p gpu
#SBATCH --mem=16g
#SBATCH --exclude=gpu[3-24],gpu[71-83],gpu[110-141],gpu[51-60]
#SBATCH -t 00:10:00
#SBATCH -c 2
#SBATCH --gres=gpu:a4000:1
#SBATCH --mail-type=END,FAIL
#SBATCH --mail-user=pkwon2@uw.edu

# Get the task command from the tasks file
task=$(sed -n "${SLURM_ARRAY_TASK_ID}p" tasks)

# Execute the task command
eval $task
"""

# Write the SBATCH script to a file
sbatch_file = os.path.join(output_dir, "digs_array_job.sh")
with open(sbatch_file, "w") as file:
    file.write(sbatch_header)

print(f"SBATCH script '{sbatch_file}' has been created.")

# Command to submit the job array
submit_command = "sbatch -a 1-$(cat tasks | wc -l) digs_array_job.sh"
print(f"""To submit the job array, use the following command:
cd {_new_directory}/{output_dir}\n{submit_command}""")

#SBATCH -J diffusion
#SBATCH -p gpu-bf
#SBATCH --mem=16g
#SBATCH -t 00:20:00
#SBATCH -c 2
#SBATCH --gres=gpu
#SBATCH --exclude=gpu[3-24],gpu[71-83],gpu[110-141],gpu[51-60]
#SBATCH --mail-type=END,FAIL
#SBATCH --mail-user=pkwon2@uw.edu

SBATCH script './1_diffusion/digs_array_job.sh' has been created.
To submit the job array, use the following command:
cd /home/pkwon2/arm_maker/./1_diffusion/
sbatch -a 1-$(cat tasks | wc -l) digs_array_job.sh


In [27]:
#!/usr/bin/env python3

import os
import glob

# Set the path to the checkpoint
ckpt_3template = '/projects/ml/aa_template/checkpoints/refine/train_session2023-11-04_1699130567.230787/BFF_3.pt'

# Set the path to the script you want to run
script = '/projects/small-molecule/rf_diffusion_pseudocycle/rf_diffusion/run_inference.py'

# Number of designs
num_designs = 2  # Adjust as needed

# Directory where you want the refine_tasks file
tasks_dir = './1_diffusion/'

# Directory containing the .pdb files relative to tasks_dir
pdb_directory = 'outputs'  # Relative to tasks_dir

# Ensure the tasks directory exists
os.makedirs(tasks_dir, exist_ok=True)

# File to write the commands to
tasks_file = os.path.join(tasks_dir, "refine_tasks")

# Get the list of .pdb files by joining tasks_dir, pdb_directory, and '*.pdb'
pdb_files = glob.glob(os.path.join(tasks_dir, pdb_directory, '*.pdb'))

# Print the last two components of the pdb file paths
for pdb_file in pdb_files:
    print('/'.join(pdb_file.split('/')[-2:]))

# Open the tasks file for writing
with open(tasks_file, 'w') as file:
    # Iterate over each .pdb file
    for pdb_file in pdb_files:
        # Adjust pdb_file to be relative as per your requirement
        relative_pdb_file = '/'.join(['./'] + pdb_file.split('/')[-2:])

        # Construct the command
        command = (
            f"/software/containers/SE3nv.sif {script} "
            "--config-name=aa "
            f"inference.ckpt_path='{ckpt_3template}' "
            "inference.model_runner='NRBStyleSelfCond' "
            f"inference.num_designs={num_designs} "
            "inference.two_template=True "
            "inference.three_template=True "
            "diffuser.T=50 "
            "inference.motif_only_2d=True "
            "inference.supply_motif_seq=True "
            "diffuser.so3_type='random' "
            "diffuser.eucl_type='gaussian' "
            "inference.refine_recycles=4 "
            "model.symmsub_k=1 "
            "inference.refine=True "
            f"inference.input_pdb='{relative_pdb_file}' "
            "model.pseudo_cycle=False "
            "inference.pseudo_symmetry='c1' "
            "model.symmetrize_repeats=False "
            "inference.align_px0_motif=False"
        )

        # Write the command to the tasks file
        file.write(command + "\n")

print(f"Commands have been written to {tasks_file}")

outputs/c2_tile_b_value1_104_ckpt_BFF_0_1_refined_0.pdb
outputs/c2_tile_b_value1_98_ckpt_BFF_0_0_refined_1.pdb
outputs/c2_tile_b_value1_91_ckpt_BFF_0_2_refined_0.pdb
outputs/c2_tile_b_value1_94_ckpt_BFF_0_3_refined_0.pdb
outputs/c2_tile_b_value1_101_ckpt_BFF_0_0_refined_0.pdb
outputs/c2_tile_b_value1_102_ckpt_BFF_0_0_refined_1.pdb
outputs/c2_tile_b_value1_89_ckpt_BFF_0_1_refined_1.pdb
outputs/c2_tile_b_value1_97_ckpt_BFF_0_3_refined_1.pdb
outputs/c2_tile_b_value1_92_ckpt_BFF_0_2_refined_1.pdb
outputs/c2_tile_b_value1_107_ckpt_BFF_0_1_refined_1.pdb
outputs/c2_tile_b_value1_93_ckpt_BFF_0_0_refined_0.pdb
outputs/c2_tile_b_value1_96_ckpt_BFF_0_1_refined_0.pdb
outputs/c2_tile_b_value1_88_ckpt_BFF_0_3_refined_0.pdb
outputs/c2_tile_b_value1_103_ckpt_BFF_0_2_refined_0.pdb
outputs/c2_tile_b_value1_100_ckpt_BFF_0_2_refined_1.pdb
outputs/c2_tile_b_value1_95_ckpt_BFF_0_1_refined_1.pdb
outputs/c2_tile_b_value1_90_ckpt_BFF_0_0_refined_1.pdb
outputs/c2_tile_b_value1_89_ckpt_BFF_0_4_refined_1.pdb
outp

In [28]:
# Define the SBATCH header, change the email to your email so it doesn't ping me!
sbatch_header = """#!/bin/bash
#SBATCH -J diffusion
#SBATCH -p gpu
#SBATCH --mem=16g
#SBATCH --exclude=gpu[3-24],gpu[71-83],gpu[110-141],gpu[51-60]
#SBATCH -t 00:10:00
#SBATCH -c 2
#SBATCH --gres=gpu:a4000:1
#SBATCH --mail-type=END,FAIL
#SBATCH --mail-user=pkwon2@uw.edu

# Get the task command from the refine_tasks file
task=$(sed -n "${SLURM_ARRAY_TASK_ID}p" refine_tasks)

# Execute the task command
eval $task
"""

# Write the SBATCH script to a file
sbatch_file = os.path.join(output_dir, "digs_array_job.sh")
with open(sbatch_file, "w") as file:
    file.write(sbatch_header)

print(f"SBATCH script '{sbatch_file}' has been created.")

# Command to submit the job array
submit_command = "sbatch -a 1-$(cat refine_tasks | wc -l) digs_array_job.sh"
print(f"""To submit the job array, use the following command:
cd {_new_directory}/{output_dir}\n{submit_command}""")

SBATCH script './1_diffusion/digs_array_job.sh' has been created.
To submit the job array, use the following command:
cd /home/pkwon2/arm_maker/./1_diffusion/
sbatch -a 1-$(cat refine_tasks | wc -l) digs_array_job.sh


In [29]:
import os
import shutil

# Define the paths
output_dir = './1_diffusion/outputs/'
unrefined_dir = os.path.join(output_dir, 'unrefined')

# Create unrefined directory if it doesn't exist
if not os.path.exists(unrefined_dir):
    os.makedirs(unrefined_dir)

# Loop through files in the output directory
for file_name in os.listdir(output_dir):
    if file_name.endswith('.pdb') and 'refined' not in file_name:
        # Construct full file path
        file_path = os.path.join(output_dir, file_name)
        
        # Move file to unrefined directory
        shutil.move(file_path, unrefined_dir)

print("All unrefined .pdb files have been moved to the 'unrefined' directory.")

All unrefined .pdb files have been moved to the 'unrefined' directory.


In [30]:
base_directory = "mpnn"
create_directory_structure(base_directory)

Directory structure 'mpnn/input' and 'mpnn/output' created successfully.


In [31]:
import os
from Bio import PDB

def find_glycines_in_pdb(file_path):
    parser = PDB.PDBParser(QUIET=True)
    structure = parser.get_structure('PDB_structure', file_path)
    glycine_positions = []

    for model in structure:
        for chain in model:
            for residue in chain:
                if residue.get_resname() == 'ALA':
                    glycine_positions.append(f"{chain.id}{residue.id[1]}")

    return ' '.join(glycine_positions)

def main():
    pdb_dir = './1_diffusion/outputs'
    
    # Convert relative path to absolute path
    pdb_dir = os.path.abspath(pdb_dir)

    if not os.path.exists(pdb_dir):
        print(f"Directory {pdb_dir} does not exist.")
        return
    # Base directory for output tasks
    base_directory = "mpnn"
    os.makedirs(base_directory, exist_ok=True)

    # Define the base command
    base_command = "/software/containers/mlfold.sif /databases/mpnn/fused_mpnn/run.py " \
                   "--pdb_path {file_path} " \
                   "--out_folder outputs " \
                   "--batch_size 2 " \
                   "--number_of_batches 2 " \
                   "--bias_AA 'M:-10.0' " \
                   "--redesigned_residues '{glycines}' " \
                   "--pack_side_chains 1 " \
                   "--repack_everything 1"

    tasks_file = os.path.join(base_directory, "tasks")

    with open(tasks_file, "w") as file:
        for pdb_file in os.listdir(pdb_dir):
            if pdb_file.endswith('.pdb'):
                file_path = os.path.join(pdb_dir, pdb_file)
                glycines = find_glycines_in_pdb(file_path)
                command = base_command.format(file_path=file_path, glycines=glycines)
                file.write(f"{command}\n")

    print(f"Tasks file '{tasks_file}' has been created.")

if __name__ == "__main__":
    main()

Tasks file 'mpnn/tasks' has been created.


In [32]:
# Define the SBATCH header, change the email to your email so it doesn't ping me!
sbatch_header = """#!/bin/bash
#SBATCH -J mpnn
#SBATCH -p cpu
#SBATCH --cpus-per-task=1
#SBATCH -t 04:00:00
#SBATCH --mem=8g
#SBATCH --mail-type=END,FAIL
#SBATCH --mail-user=pkwon2@uw.edu

# Get the task command from the tasks file
task=$(sed -n "${SLURM_ARRAY_TASK_ID}p" tasks)

# Execute the task command
eval $task
"""

# Write the SBATCH script to a file
sbatch_file = os.path.join(base_directory, "digs_array_job.sh")
with open(sbatch_file, "w") as file:
    file.write(sbatch_header)

print(f"SBATCH script '{sbatch_file}' has been created.")

# Command to submit the job array
submit_command = "sbatch -a 1-$(cat tasks | wc -l) digs_array_job.sh"
print(f"""To submit the job array, use the following command:
cd {_new_directory}/{base_directory}\n{submit_command}""")

SBATCH script 'mpnn/digs_array_job.sh' has been created.
To submit the job array, use the following command:
cd /home/pkwon2/arm_maker/mpnn
sbatch -a 1-$(cat tasks | wc -l) digs_array_job.sh


In [33]:
base_directory = "superfold"
create_directory_structure(base_directory)

def main():
    pdb_dir = './mpnn/outputs/packed' #this is where your mpnn outputs are
    
    # Convert relative path to absolute path
    pdb_dir = os.path.abspath(pdb_dir)

    if not os.path.exists(pdb_dir):
        print(f"Directory {pdb_dir} does not exist.")
        return

    # Define the base command
    base_command = "/software/lab/superfold/superfold {file_path} " \
                   "--models 3 " \
                   "--initial_guess " \
                   "--out_dir outputs " \

    tasks_file = os.path.join(base_directory, "tasks")

    with open(tasks_file, "w") as file:
        for pdb_file in os.listdir(pdb_dir):
            if pdb_file.endswith('.pdb'):
                file_path = os.path.join(pdb_dir, pdb_file)
                command = base_command.format(file_path=file_path)
                file.write(f"{command}\n")

    print(f"Tasks file '{tasks_file}' has been created.")

if __name__ == "__main__":
    main()

Directory structure 'superfold/input' and 'superfold/output' created successfully.
Tasks file 'superfold/tasks' has been created.


In [34]:
# Define the SBATCH header, change the email to your email so it doesn't ping me!
sbatch_header = """#!/bin/bash
#SBATCH -J diffusion
#SBATCH -p gpu
#SBATCH --mem=16g
#SBATCH --exclude=gpu[3-24],gpu[71-83],gpu[110-141],gpu[51-60]
#SBATCH -t 00:10:00
#SBATCH -c 2
#SBATCH --gres=gpu:a4000:1
#SBATCH --mail-type=END,FAIL
#SBATCH --mail-user=pkwon2@uw.edu

# Get the task command from the tasks file
task=$(sed -n "${SLURM_ARRAY_TASK_ID}p" tasks)

# Execute the task command
eval $task
"""

# Write the SBATCH script to a file
sbatch_file = os.path.join(base_directory, "digs_array_job.sh")
with open(sbatch_file, "w") as file:
    file.write(sbatch_header)

print(f"SBATCH script '{sbatch_file}' has been created.")

# Command to submit the job array
submit_command = "sbatch -a 1-$(cat tasks | wc -l) digs_array_job.sh"
print(f"""To submit the job array, use the following command:
cd {_new_directory}/{base_directory}\n{submit_command}""")

SBATCH script 'superfold/digs_array_job.sh' has been created.
To submit the job array, use the following command:
cd /home/pkwon2/arm_maker/superfold
sbatch -a 1-$(cat tasks | wc -l) digs_array_job.sh


In [35]:
import os
import pandas as pd
import numpy as np
from Bio.PDB import PDBParser, Superimposer
import shutil

# Define directories
packed_dir = './mpnn/outputs/packed'
superfold_outputs_dir = './superfold/outputs'
rmsd_filtered_dir = './superfold/rmsd_filtered'

# Create the filtered directory if it doesn't exist
os.makedirs(rmsd_filtered_dir, exist_ok=True)

# Initialize PDB parser
parser = PDBParser()

def calculate_rmsd(file1, file2):
    """Calculate RMSD between two PDB files."""
    structure1 = parser.get_structure('structure1', file1)
    structure2 = parser.get_structure('structure2', file2)
    
    # Extract atoms
    atoms1 = list(structure1.get_atoms())
    atoms2 = list(structure2.get_atoms())
    
    # Superimpose structures
    super_imposer = Superimposer()
    super_imposer.set_atoms(atoms1, atoms2)
    
    return super_imposer.rms

# Dataframe to store paths and RMSD values
data = []

# Iterate over files in the packed directory
for packed_pdb in os.listdir(packed_dir):
    if packed_pdb.endswith('.pdb'):
        # print(packed_pdb)
        # Construct corresponding file names in superfold directory
        superfold_pdb = packed_pdb.replace('.pdb', '_model_3_ptm_seed_0_unrelaxed.pdb')
        print(superfold_pdb)
        superfold_json = superfold_pdb.replace('unrelaxed.pdb', 'prediction_results.json')
        
        superfold_pdb_path = os.path.join(superfold_outputs_dir, superfold_pdb)
        superfold_json_path = os.path.join(superfold_outputs_dir, superfold_json)
        
        # Ensure the corresponding superfold pdb and json files exist
        if os.path.exists(superfold_pdb_path) and os.path.exists(superfold_json_path):
            packed_pdb_path = os.path.join(packed_dir, packed_pdb)
            
            # Calculate RMSD
            rmsd = calculate_rmsd(packed_pdb_path, superfold_pdb_path)
            
            # Check RMSD threshold
            if rmsd < 1.5:
                # Copy pdb and json files to rmsd_filtered directory
                shutil.copy(superfold_pdb_path, rmsd_filtered_dir)
                shutil.copy(superfold_json_path, rmsd_filtered_dir)
                
                # Add to dataframe
                data.append([packed_pdb_path, superfold_pdb_path, rmsd])

# Convert to DataFrame and save to CSV
df = pd.DataFrame(data, columns=['Packed PDB Path', 'Superfold PDB Path', 'RMSD'])
csv_path = os.path.join(rmsd_filtered_dir, 'rmsd_filtered.csv')
df.to_csv(csv_path, index=False)

print(f'RMSD calculation and file filtering complete. Results saved to {csv_path}')

c2_tile_b_value1_88_ckpt_BFF_0_0_refined_1_packed_2_1_model_3_ptm_seed_0_unrelaxed.pdb
c2_tile_b_value1_104_ckpt_BFF_0_0_refined_1_packed_1_1_model_3_ptm_seed_0_unrelaxed.pdb
c2_tile_b_value1_100_ckpt_BFF_0_0_refined_1_packed_3_1_model_3_ptm_seed_0_unrelaxed.pdb


/home/pkwon2/.local/lib/python3.10/site-packages/Bio/PDB/PDBParser.py:395: PDBConstructionWarning: Ignoring unrecognized record 'END' at line 1941
  warnings.warn(
/home/pkwon2/.local/lib/python3.10/site-packages/Bio/PDB/PDBParser.py:395: PDBConstructionWarning: Ignoring unrecognized record 'END' at line 1941
  warnings.warn(
/home/pkwon2/.local/lib/python3.10/site-packages/Bio/PDB/PDBParser.py:395: PDBConstructionWarning: Ignoring unrecognized record 'END' at line 2089
  warnings.warn(
/home/pkwon2/.local/lib/python3.10/site-packages/Bio/PDB/PDBParser.py:395: PDBConstructionWarning: Ignoring unrecognized record 'END' at line 2089
  warnings.warn(
/home/pkwon2/.local/lib/python3.10/site-packages/Bio/PDB/PDBParser.py:395: PDBConstructionWarning: Ignoring unrecognized record 'END' at line 2034
  warnings.warn(
/home/pkwon2/.local/lib/python3.10/site-packages/Bio/PDB/PDBParser.py:395: PDBConstructionWarning: Ignoring unrecognized record 'END' at line 2034
  warnings.warn(
/home/pkwon2/.lo

c2_tile_b_value1_102_ckpt_BFF_0_0_refined_1_packed_2_1_model_3_ptm_seed_0_unrelaxed.pdb
c2_tile_b_value1_100_ckpt_BFF_0_1_refined_0_packed_4_1_model_3_ptm_seed_0_unrelaxed.pdb
c2_tile_b_value1_92_ckpt_BFF_0_1_refined_1_packed_4_1_model_3_ptm_seed_0_unrelaxed.pdb


/home/pkwon2/.local/lib/python3.10/site-packages/Bio/PDB/PDBParser.py:395: PDBConstructionWarning: Ignoring unrecognized record 'END' at line 1986
  warnings.warn(


c2_tile_b_value1_90_ckpt_BFF_0_0_refined_0_packed_2_1_model_3_ptm_seed_0_unrelaxed.pdb


...

/home/pkwon2/.local/lib/python3.10/site-packages/Bio/PDB/PDBParser.py:395: PDBConstructionWarning: Ignoring unrecognized record 'END' at line 2077
  warnings.warn(
/home/pkwon2/.local/lib/python3.10/site-packages/Bio/PDB/PDBParser.py:395: PDBConstructionWarning: Ignoring unrecognized record 'END' at line 2077
  warnings.warn(
/home/pkwon2/.local/lib/python3.10/site-packages/Bio/PDB/PDBParser.py:395: PDBConstructionWarning: Ignoring unrecognized record 'END' at line 2010
  warnings.warn(
/home/pkwon2/.local/lib/python3.10/site-packages/Bio/PDB/PDBParser.py:395: PDBConstructionWarning: Ignoring unrecognized record 'END' at line 2010
  warnings.warn(
/home/pkwon2/.local/lib/python3.10/site-packages/Bio/PDB/PDBParser.py:395: PDBConstructionWarning: Ignoring unrecognized record 'END' at line 1997
  warnings.warn(
/home/pkwon2/.local/lib/python3.10/site-packages/Bio/PDB/PDBParser.py:395: PDBConstructionWarning: Ignoring unrecognized record 'END' at line 1997
  warnings.warn(


c2_tile_b_value1_101_ckpt_BFF_0_0_refined_1_packed_4_1_model_3_ptm_seed_0_unrelaxed.pdb
c2_tile_b_value1_93_ckpt_BFF_0_0_refined_0_packed_4_1_model_3_ptm_seed_0_unrelaxed.pdb
c2_tile_b_value1_99_ckpt_BFF_0_0_refined_0_packed_1_1_model_3_ptm_seed_0_unrelaxed.pdb


/home/pkwon2/.local/lib/python3.10/site-packages/Bio/PDB/PDBParser.py:395: PDBConstructionWarning: Ignoring unrecognized record 'END' at line 1990
  warnings.warn(
/home/pkwon2/.local/lib/python3.10/site-packages/Bio/PDB/PDBParser.py:395: PDBConstructionWarning: Ignoring unrecognized record 'END' at line 1990
  warnings.warn(
/home/pkwon2/.local/lib/python3.10/site-packages/Bio/PDB/PDBParser.py:395: PDBConstructionWarning: Ignoring unrecognized record 'END' at line 1931
  warnings.warn(
/home/pkwon2/.local/lib/python3.10/site-packages/Bio/PDB/PDBParser.py:395: PDBConstructionWarning: Ignoring unrecognized record 'END' at line 1931
  warnings.warn(
/home/pkwon2/.local/lib/python3.10/site-packages/Bio/PDB/PDBParser.py:395: PDBConstructionWarning: Ignoring unrecognized record 'END' at line 1987
  warnings.warn(
/home/pkwon2/.local/lib/python3.10/site-packages/Bio/PDB/PDBParser.py:395: PDBConstructionWarning: Ignoring unrecognized record 'END' at line 1987
  warnings.warn(


c2_tile_b_value1_91_ckpt_BFF_0_1_refined_1_packed_2_1_model_3_ptm_seed_0_unrelaxed.pdb
c2_tile_b_value1_93_ckpt_BFF_0_1_refined_1_packed_3_1_model_3_ptm_seed_0_unrelaxed.pdb
c2_tile_b_value1_97_ckpt_BFF_0_1_refined_1_packed_1_1_model_3_ptm_seed_0_unrelaxed.pdb
RMSD calculation and file filtering complete. Results saved to ./superfold/rmsd_filtered/rmsd_filtered.csv


/home/pkwon2/.local/lib/python3.10/site-packages/Bio/PDB/PDBParser.py:395: PDBConstructionWarning: Ignoring unrecognized record 'END' at line 2021
  warnings.warn(
/home/pkwon2/.local/lib/python3.10/site-packages/Bio/PDB/PDBParser.py:395: PDBConstructionWarning: Ignoring unrecognized record 'END' at line 2021
  warnings.warn(


In [36]:
import os
import json

# Define the directory containing the .json and .pdb files
directory = './superfold/rmsd_filtered'

# Loop through all files in the directory
for filename in os.listdir(directory):
    # Check if the file is a .json file
    if filename.endswith('prediction_results.json'):
        json_path = os.path.join(directory, filename)
        
        # Read the .json file
        with open(json_path, 'r') as json_file:
            data = json.load(json_file)
            
            # Get the mean_plddt value
            mean_plddt = data.get('mean_plddt', 1.0)  # Default to 1.0 if not found
            
            # Print the mean_plddt value and the .json filename
            # print(f'{filename}: mean_plddt = {mean_plddt}')
            
            # Check if mean_plddt is less than 80
            if mean_plddt < 90:
                # Construct the corresponding .pdb filename
                pdb_filename = filename.replace('prediction_results.json', 'unrelaxed.pdb')
                # print(pdb_filename)
                pdb_path = os.path.join(directory, pdb_filename)
                # print(pdb_path)
                # Delete the .pdb file if it exists
                if os.path.exists(pdb_path):
                    os.remove(pdb_path)
                    print(f'Deleted {pdb_path}')
                else:
                    print(f'{pdb_path} does not exist')
                
                # Delete the .json file
                os.remove(json_path)
                print(f'Deleted {json_path}')

print('Processing complete.')

Processing complete.


In [37]:
import os
import json
from collections import defaultdict

# Define the directory containing the .json and .pdb files
directory = './superfold/rmsd_filtered'

# Initialize a dictionary to hold groups of files
file_groups = defaultdict(list)

# Function to extract the prefix based on the last instance of 'packed'
def get_prefix(filename):
    parts = filename.split('value1')
    if len(parts) > 1:
        return 'value1'.join(parts[:-1]) + 'value1'
    return filename

# Loop through all files in the directory
for filename in os.listdir(directory):
    if filename.endswith('prediction_results.json'):
        prefix = get_prefix(filename)
        json_path = os.path.join(directory, filename)
        
        # Read the .json file
        with open(json_path, 'r') as json_file:
            data = json.load(json_file)
            
            # Get the mean_plddt value
            mean_plddt = data.get('mean_plddt', 0.0)  # Default to 0.0 if not found
            
            # Append the file info to the group
            file_groups[prefix].append((mean_plddt, json_path, filename))

# Function to delete a file
def delete_file(file_path):
    if os.path.exists(file_path):
        os.remove(file_path)
        print(f'Deleted {file_path}')
    else:
        print(f'{file_path} does not exist')

# Iterate through each group
for prefix, files in file_groups.items():
    # Sort the files by mean_plddt in descending order
    files.sort(reverse=True, key=lambda x: x[0])
    
    # Get the highest n files (modify 'n' to choose highest 1, 2, 3, etc.)
    n = 3  # For highest 1 file per group, change to 3 for highest 3 per group
    highest_files = files[:n]
    
    # Files to keep
    highest_files_set = set(f[1] for f in highest_files)
    
    # Iterate through the files in the group
    for _, json_path, filename in files:
        if json_path not in highest_files_set:
            # Delete the .json file
            delete_file(json_path)
            
            # Construct the corresponding .pdb filename and delete it
            pdb_filename = filename.replace('prediction_results.json', 'unrelaxed.pdb')
            pdb_path = os.path.join(directory, pdb_filename)
            delete_file(pdb_path)

print('Processing complete.')

Deleted ./superfold/rmsd_filtered/c2_tile_b_value1_98_ckpt_BFF_0_2_refined_0_packed_4_1_model_3_ptm_seed_0_prediction_results.json
Deleted ./superfold/rmsd_filtered/c2_tile_b_value1_98_ckpt_BFF_0_2_refined_0_packed_4_1_model_3_ptm_seed_0_unrelaxed.pdb
Deleted ./superfold/rmsd_filtered/c2_tile_b_value1_103_ckpt_BFF_0_2_refined_0_packed_1_1_model_3_ptm_seed_0_prediction_results.json
Deleted ./superfold/rmsd_filtered/c2_tile_b_value1_103_ckpt_BFF_0_2_refined_0_packed_1_1_model_3_ptm_seed_0_unrelaxed.pdb
Deleted ./superfold/rmsd_filtered/c2_tile_b_value1_101_ckpt_BFF_0_0_refined_1_packed_1_1_model_3_ptm_seed_0_prediction_results.json
Deleted ./superfold/rmsd_filtered/c2_tile_b_value1_101_ckpt_BFF_0_0_refined_1_packed_1_1_model_3_ptm_seed_0_unrelaxed.pdb
Deleted ./superfold/rmsd_filtered/c2_tile_b_value1_92_ckpt_BFF_0_1_refined_0_packed_1_1_model_3_ptm_seed_0_prediction_results.json
Deleted ./superfold/rmsd_filtered/c2_tile_b_value1_92_ckpt_BFF_0_1_refined_0_packed_1_1_model_3_ptm_seed_0_un

In [2]:
import os
from Bio.PDB import PDBParser, PDBIO
from Bio.PDB.Model import Model
from Bio.PDB.Structure import Structure
from Bio.PDB.Chain import Chain

# Directory containing the PDB files
pdb_dir = './superfold/rmsd_filtered/'

# Initialize PDB parser and PDBIO
parser = PDBParser(QUIET=True)
io = PDBIO()

# Iterate over PDB files in the directory
for pdb_filename in os.listdir(pdb_dir):
    if pdb_filename.endswith('.pdb'):
        pdb_path = os.path.join(pdb_dir, pdb_filename)
        print(f'Processing {pdb_path}')

        # Parse the structure
        structure = parser.get_structure('structure', pdb_path)

        # Assume only one model per structure
        model = structure[0]

        # Get chains and their lengths
        chains = list(model.get_chains())
        chain_lengths = []
        for chain in chains:
            residues = list(chain.get_residues())
            num_residues = len(residues)
            chain_lengths.append((chain, num_residues))

        # Find the chain with the maximum number of residues
        longest_chain, max_length = max(chain_lengths, key=lambda x: x[1])

        # Create a new model to store only the longest chain
        new_model = Model(model.id)

        # Create a new chain with ID 'A'
        new_chain = Chain('A')

        # Copy residues from the longest chain to the new chain
        for residue in longest_chain.get_residues():
            new_chain.add(residue)

        # Add the new chain to the new model
        new_model.add(new_chain)

        # Create a new structure with the new model
        new_structure = Structure(structure.id)
        new_structure.add(new_model)

        # Save the new structure back to the same PDB file
        io.set_structure(new_structure)
        io.save(pdb_path)
        print(f'Saved updated structure to {pdb_path}')

        # Rename the file by replacing 'good_cage_value' with 'good_cage_tile_a_value'
        new_pdb_filename = pdb_filename.replace('good_cage_value', 'good_cage_tile_a_value')
        new_pdb_path = os.path.join(pdb_dir, new_pdb_filename)

        # Rename the file
        os.rename(pdb_path, new_pdb_path)
        print(f'Renamed {pdb_filename} to {new_pdb_filename}')

Processing ./superfold/rmsd_filtered/c6_value1_129_ckpt_BFF_0_2_refined_0_packed_1_1_model_3_ptm_seed_0_unrelaxed.pdb
Saved updated structure to ./superfold/rmsd_filtered/c6_value1_129_ckpt_BFF_0_2_refined_0_packed_1_1_model_3_ptm_seed_0_unrelaxed.pdb
Renamed c6_value1_129_ckpt_BFF_0_2_refined_0_packed_1_1_model_3_ptm_seed_0_unrelaxed.pdb to c6_value1_129_ckpt_BFF_0_2_refined_0_packed_1_1_model_3_ptm_seed_0_unrelaxed.pdb
Processing ./superfold/rmsd_filtered/combined_fd31_r6_pk_model_5_ptm_seed_0_unrelaxed_y22_C4_y-65_C2_LHD101_model_5_ptm_seed_0_unrelaxed_tile_a_value1_103_ckpt_BFF_0_2_refined_1_packed_4_1_model_3_ptm_seed_0_unrelaxed.pdb
Saved updated structure to ./superfold/rmsd_filtered/combined_fd31_r6_pk_model_5_ptm_seed_0_unrelaxed_y22_C4_y-65_C2_LHD101_model_5_ptm_seed_0_unrelaxed_tile_a_value1_103_ckpt_BFF_0_2_refined_1_packed_4_1_model_3_ptm_seed_0_unrelaxed.pdb
Renamed combined_fd31_r6_pk_model_5_ptm_seed_0_unrelaxed_y22_C4_y-65_C2_LHD101_model_5_ptm_seed_0_unrelaxed_tile_a_

In [18]:
import os
from Bio import SeqIO
from Bio.PDB import PDBParser, Polypeptide
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

input_dir = './superfold/rmsd_filtered'
output_dir = './superfold/fasta'

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

def get_chain_a_sequence(pdb_file, tile_type):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure('structure', pdb_file)
    
    chain_a_seq = ''
    for model in structure:
        for chain in model:
            if chain.id == 'A':
                for residue in chain:
                    if residue.get_id()[0] == ' ':
                        chain_a_seq += Polypeptide.three_to_one(residue.resname)
    if tile_type == 'tile':
        # Remove the first 52 residues
        return chain_a_seq[52:]
    else:
        # Replace the first residues with the given sequence
        first_replacement = 'DRELLKEAIETVRELLEKLLERNPNDKELIKKILEIVEKIFELTYKVAVAEK'
        len_first_replacement = len(first_replacement)
        
        # Remove the last 52 residues from the original sequence
        # and get the middle sequence starting from len_first_replacement
        middle_seq = chain_a_seq[len_first_replacement:-52]
        
        # Append the specified last replacement
        last_replacement = 'DRELLKEAIETVRELLEKLLERNPN'
        
        # Construct the final sequence
        return first_replacement + middle_seq + last_replacement

def extract_id_parts(filename, is_tile):
    base_name = os.path.basename(filename)
    name_without_ext = os.path.splitext(base_name)[0]
    parts = name_without_ext.split('_')
    
    if not is_tile:
        # For non-tile PDBs
        c_value = parts[0]
        value1_value = ''
        bff_packed = ''
        if 'value1' in parts:
            value1_index = parts.index('value1')
            if value1_index + 1 < len(parts):
                value1_value = parts[value1_index + 1]
        # Extract from 'BFF' up to 'packed'
        if 'BFF' in parts:
            bff_index = parts.index('BFF')
            end_index = len(parts)
            for i in range(bff_index + 1, len(parts)):
                if parts[i] in ('model', 'ptm', 'seed', 'unrelaxed', 'tile'):
                    end_index = i
                    break
            bff_packed_parts = parts[bff_index:end_index]
            bff_packed = '_'.join(bff_packed_parts)
        return {'c_value': c_value, 'value1': value1_value, 'bff_packed': bff_packed}
    else:
        # For tile PDBs
        name_lower = name_without_ext.lower()
        tile_type = 'tile_a' if 'tile_a' in name_lower else 'tile_b' if 'tile_b' in name_lower else ''
        value1_value = ''
        bff_packed = ''
        if 'value1' in parts:
            value1_index = parts.index('value1')
            if value1_index + 1 < len(parts):
                value1_value = parts[value1_index + 1]
        # Extract from 'BFF' up to 'packed'
        if 'BFF' in parts:
            bff_index = parts.index('BFF')
            end_index = len(parts)
            for i in range(bff_index + 1, len(parts)):
                if parts[i] in ('model', 'ptm', 'seed', 'unrelaxed', 'tile'):
                    end_index = i
                    break
            bff_packed_parts = parts[bff_index:end_index]
            bff_packed = '_'.join(bff_packed_parts)
        return {'tile_type': tile_type, 'value1': value1_value, 'bff_packed': bff_packed}

def pdb_to_fasta(pdb_file_1, pdb_file_2, fasta_file, record_id):
    seq_1 = get_chain_a_sequence(pdb_file_1, 'non-tile')
    seq_2 = get_chain_a_sequence(pdb_file_2, 'tile')
    
    final_sequence = seq_1 + seq_2
    
    seq = Seq(final_sequence)
    record = SeqRecord(seq, id=record_id, description="")
    
    # Print the record
    print(record)
    
    # Write to fasta file
    SeqIO.write(record, fasta_file, 'fasta')

# Collect files
tile_a_files = [f for f in os.listdir(input_dir) if f.endswith('.pdb') and 'tile_a' in f]
tile_b_files = [f for f in os.listdir(input_dir) if f.endswith('.pdb') and 'tile_b' in f]
non_tile_files = [f for f in os.listdir(input_dir) if f.endswith('.pdb') and 'tile' not in f]

# Process tile_a + non-tile pairs
for tile_file in tile_a_files:
    for non_tile_file in non_tile_files:
        pdb_file_path_tile = os.path.join(input_dir, tile_file)
        pdb_file_path_non_tile = os.path.join(input_dir, non_tile_file)
        
        # Extract id parts
        non_tile_id_parts = extract_id_parts(non_tile_file, is_tile=False)
        tile_id_parts = extract_id_parts(tile_file, is_tile=True)
        
        id = f"{non_tile_id_parts['c_value']}_{non_tile_id_parts['value1']}_{non_tile_id_parts['bff_packed']}_{tile_id_parts['tile_type']}_{tile_id_parts['value1']}_{tile_id_parts['bff_packed']}"
        
        # Construct fasta filename
        fasta_filename = id + '.fasta'
        fasta_file_path = os.path.join(output_dir, fasta_filename)
        
        # Print the pair
        print(f"Processing pair: {tile_file} + {non_tile_file}")
        
        pdb_to_fasta(pdb_file_path_non_tile, pdb_file_path_tile, fasta_file_path, id)

# Process tile_b + non-tile pairs
for tile_file in tile_b_files:
    for non_tile_file in non_tile_files:
        pdb_file_path_tile = os.path.join(input_dir, tile_file)
        pdb_file_path_non_tile = os.path.join(input_dir, non_tile_file)
        
        # Extract id parts
        non_tile_id_parts = extract_id_parts(non_tile_file, is_tile=False)
        tile_id_parts = extract_id_parts(tile_file, is_tile=True)
        
        id = f"{non_tile_id_parts['c_value']}_{non_tile_id_parts['value1']}_{non_tile_id_parts['bff_packed']}_{tile_id_parts['tile_type']}_{tile_id_parts['value1']}_{tile_id_parts['bff_packed']}"
        
        # Construct fasta filename
        fasta_filename = id + '.fasta'
        fasta_file_path = os.path.join(output_dir, fasta_filename)
        
        # Print the pair
        print(f"Processing pair: {tile_file} + {non_tile_file}")
        
        pdb_to_fasta(pdb_file_path_non_tile, pdb_file_path_tile, fasta_file_path, id)

print("Conversion completed. FASTA files are located in:", output_dir)

Processing pair: combined_fd31_r6_pk_model_5_ptm_seed_0_unrelaxed_y22_C4_y-65_C2_LHD101_model_5_ptm_seed_0_unrelaxed_tile_a_value1_103_ckpt_BFF_0_2_refined_1_packed_4_1_model_3_ptm_seed_0_unrelaxed.pdb + c6_value1_129_ckpt_BFF_0_2_refined_0_packed_1_1_model_3_ptm_seed_0_unrelaxed.pdb
ID: c6_129_BFF_0_2_refined_0_packed_1_1_tile_a_103_BFF_0_2_refined_1_packed_4_1
Name: <unknown name>
Number of features: 0
Seq('DRELLKEAIETVRELLEKLLERNPNDKELIKKILEIVEKIFELTYKVAVAEKDA...VRE')
Processing pair: combined_fd31_r6_pk_model_5_ptm_seed_0_unrelaxed_y22_C4_y-65_C2_LHD101_model_5_ptm_seed_0_unrelaxed_tile_a_value1_103_ckpt_BFF_0_2_refined_1_packed_4_1_model_3_ptm_seed_0_unrelaxed.pdb + c4_value1_138_ckpt_BFF_0_1_refined_0_packed_2_1_model_3_ptm_seed_0_unrelaxed.pdb
ID: c4_138_BFF_0_1_refined_0_packed_2_1_tile_a_103_BFF_0_2_refined_1_packed_4_1
Name: <unknown name>
Number of features: 0
Seq('DRELLKEAIETVRELLEKLLERNPNDKELIKKILEIVEKIFELTYKVAVAEKGK...VRE')
Processing pair: combined_fd31_r6_pk_model_5_ptm

In [19]:
base_directory = "superfold"
create_directory_structure(base_directory)

array_outputs_dir = os.path.join(base_directory, 'array_outputs')
os.makedirs(array_outputs_dir, exist_ok=True)

def main():
    pdb_dir = './superfold/fasta' #this is where your mpnn outputs are
    
    # Convert relative path to absolute path
    pdb_dir = os.path.abspath(pdb_dir)

    if not os.path.exists(pdb_dir):
        print(f"Directory {pdb_dir} does not exist.")
        return

    # Define the base command
    base_command = "/software/lab/superfold/superfold {file_path} " \
                   "--models 3 " \
                   "--out_dir array_outputs " \

    tasks_file = os.path.join(base_directory, "array_tasks")

    with open(tasks_file, "w") as file:
        for pdb_file in os.listdir(pdb_dir):
            if pdb_file.endswith('.fasta'):
                file_path = os.path.join(pdb_dir, pdb_file)
                command = base_command.format(file_path=file_path)
                file.write(f"{command}\n")

    print(f"Tasks file '{tasks_file}' has been created.")

if __name__ == "__main__":
    main()

Directory structure 'superfold/input' and 'superfold/output' created successfully.
Tasks file 'superfold/array_tasks' has been created.


In [20]:
# Define the SBATCH header, change the email to your email so it doesn't ping me!
sbatch_header = """#!/bin/bash
#SBATCH -J alphafold
#SBATCH -p gpu
#SBATCH --mem=16g
#SBATCH --exclude=gpu[3-24],gpu[71-83],gpu[110-141],gpu[51-60]
#SBATCH -t 00:20:00
#SBATCH -c 2
#SBATCH --gres=gpu:a4000:1
#SBATCH --mail-type=END,FAIL
#SBATCH --mail-user=pkwon2@uw.edu

# Get the array_tasks command from the array_tasks file
task=$(sed -n "${SLURM_ARRAY_TASK_ID}p" array_tasks)

# Execute the task command
eval $task
"""

# Write the SBATCH script to a file
sbatch_file = os.path.join(base_directory, "digs_array_job.sh")
with open(sbatch_file, "w") as file:
    file.write(sbatch_header)

print(f"SBATCH script '{sbatch_file}' has been created.")

# Command to submit the job array
submit_command = "sbatch -a 1-$(cat array_tasks | wc -l) digs_array_job.sh"
print(f"""To submit the job array, use the following command:
cd {_new_directory}/{base_directory}\n{submit_command}""")

SBATCH script 'superfold/digs_array_job.sh' has been created.
To submit the job array, use the following command:
cd /home/pkwon2/arm_maker/superfold
sbatch -a 1-$(cat array_tasks | wc -l) digs_array_job.sh


In [21]:
import os
import json

# Define the directory containing the .json and .pdb files
directory = './superfold/array_outputs'

# Loop through all files in the directory
for filename in os.listdir(directory):
    # Check if the file is a .json file
    if filename.endswith('prediction_results.json'):
        json_path = os.path.join(directory, filename)
        
        # Read the .json file
        with open(json_path, 'r') as json_file:
            data = json.load(json_file)
            
            # Get the mean_plddt value
            mean_plddt = data.get('mean_plddt', 1.0)  # Default to 1.0 if not found
            
            # Print the mean_plddt value and the .json filename
            # print(f'{filename}: mean_plddt = {mean_plddt}')
            
            # Check if mean_plddt is less than 80
            if mean_plddt < 80:
                # Construct the corresponding .pdb filename
                pdb_filename = filename.replace('prediction_results.json', 'unrelaxed.pdb')
                # print(pdb_filename)
                pdb_path = os.path.join(directory, pdb_filename)
                # print(pdb_path)
                # Delete the .pdb file if it exists
                if os.path.exists(pdb_path):
                    os.remove(pdb_path)
                    print(f'Deleted {pdb_path}')
                else:
                    print(f'{pdb_path} does not exist')
                
                # Delete the .json file
                os.remove(json_path)
                print(f'Deleted {json_path}')

print('Processing complete.')

Deleted ./superfold/array_outputs/c4_138_BFF_0_1_refined_0_packed_2_1_tile_b_107_BFF_0_2_refined_1_packed_1_1_model_3_ptm_seed_0_unrelaxed.pdb
Deleted ./superfold/array_outputs/c4_138_BFF_0_1_refined_0_packed_2_1_tile_b_107_BFF_0_2_refined_1_packed_1_1_model_3_ptm_seed_0_prediction_results.json
Deleted ./superfold/array_outputs/c4_132_BFF_0_1_refined_0_packed_3_1_tile_a_103_BFF_0_2_refined_1_packed_4_1_model_3_ptm_seed_0_unrelaxed.pdb
Deleted ./superfold/array_outputs/c4_132_BFF_0_1_refined_0_packed_3_1_tile_a_103_BFF_0_2_refined_1_packed_4_1_model_3_ptm_seed_0_prediction_results.json
Deleted ./superfold/array_outputs/c4_138_BFF_0_4_refined_1_packed_3_1_tile_a_100_BFF_0_1_refined_0_packed_4_1_model_3_ptm_seed_0_unrelaxed.pdb
Deleted ./superfold/array_outputs/c4_138_BFF_0_4_refined_1_packed_3_1_tile_a_100_BFF_0_1_refined_0_packed_4_1_model_3_ptm_seed_0_prediction_results.json
Deleted ./superfold/array_outputs/c4_138_BFF_0_1_refined_0_packed_2_1_tile_a_103_BFF_0_2_refined_1_packed_4_1_mo

In [10]:
import os
import json
from collections import defaultdict

# Define the directory containing the .json and .pdb files
directory = './superfold/array_outputs'

# Initialize a dictionary to hold groups of files
file_groups = defaultdict(list)

# Function to extract the prefix based on the last instance of 'model'
def get_prefix(filename):
    parts = filename.split('model')
    if len(parts) > 1:
        return 'model'.join(parts[:-1]) + 'model'
    return filename

# Loop through all files in the directory
for filename in os.listdir(directory):
    if filename.endswith('prediction_results.json'):
        prefix = get_prefix(filename)
        json_path = os.path.join(directory, filename)
        
        # Read the .json file
        with open(json_path, 'r') as json_file:
            data = json.load(json_file)
            
            # Get the mean_plddt value
            mean_plddt = data.get('mean_plddt', 0.0)  # Default to 0.0 if not found
            
            # Append the file info to the group
            file_groups[prefix].append((mean_plddt, json_path, filename))

# Function to delete a file
def delete_file(file_path):
    if os.path.exists(file_path):
        os.remove(file_path)
        print(f'Deleted {file_path}')
    else:
        print(f'{file_path} does not exist')

# Iterate through each group
for prefix, files in file_groups.items():
    # Sort the files by mean_plddt in descending order
    files.sort(reverse=True, key=lambda x: x[0])
    
    # Get the highest n files (modify 'n' to choose highest 1, 2, 3, etc.)
    n = 1  # For highest 1 file per group, change to 3 for highest 3 per group
    highest_files = files[:n]
    
    # Files to keep
    highest_files_set = set(f[1] for f in highest_files)
    
    # Iterate through the files in the group
    for _, json_path, filename in files:
        if json_path not in highest_files_set:
            # Delete the .json file
            delete_file(json_path)
            
            # Construct the corresponding .pdb filename and delete it
            pdb_filename = filename.replace('prediction_results.json', 'unrelaxed.pdb')
            pdb_path = os.path.join(directory, pdb_filename)
            delete_file(pdb_path)

print('Processing complete.')

Deleted ./superfold/array_outputs/C3_y126_10_tile_a_value1_103_ckpt_BFF_0_103_ckpt_BFF_0_2_refined_0_model_2_ptm_seed_0_prediction_results.json
Deleted ./superfold/array_outputs/C3_y126_10_tile_a_value1_103_ckpt_BFF_0_103_ckpt_BFF_0_2_refined_0_model_2_ptm_seed_0_unrelaxed.pdb
Deleted ./superfold/array_outputs/C3_y126_10_tile_a_value1_103_ckpt_BFF_0_103_ckpt_BFF_0_2_refined_0_model_4_ptm_seed_0_prediction_results.json
Deleted ./superfold/array_outputs/C3_y126_10_tile_a_value1_103_ckpt_BFF_0_103_ckpt_BFF_0_2_refined_0_model_4_ptm_seed_0_unrelaxed.pdb
Deleted ./superfold/array_outputs/C3_y126_10_tile_a_value1_103_ckpt_BFF_0_103_ckpt_BFF_0_2_refined_0_model_3_ptm_seed_0_prediction_results.json
Deleted ./superfold/array_outputs/C3_y126_10_tile_a_value1_103_ckpt_BFF_0_103_ckpt_BFF_0_2_refined_0_model_3_ptm_seed_0_unrelaxed.pdb
Deleted ./superfold/array_outputs/C3_y126_10_tile_a_value1_103_ckpt_BFF_0_103_ckpt_BFF_0_2_refined_1_model_4_ptm_seed_0_prediction_results.json
Deleted ./superfold/ar

In [14]:
import os
from Bio import PDB
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO

def extract_sequences_from_pdb(pdb_file):
    parser = PDB.PDBParser(QUIET=True)
    structure = parser.get_structure('structure', pdb_file)
    
    sequences = []
    for model in structure:
        for chain in model:
            seq = []
            for residue in chain:
                if PDB.is_aa(residue):
                    seq.append(PDB.Polypeptide.three_to_one(residue.resname))
            sequence = ''.join(seq)
            chain_id = chain.id
            sequences.append((chain_id, sequence))
    
    return sequences

def write_fasta(sequences, output_file):
    records = []
    for pdb_id, chain_id, sequence in sequences:
        record = SeqRecord(Seq(sequence), id=f"{pdb_id}_{chain_id}", description="")
        records.append(record)
    SeqIO.write(records, output_file, "fasta")

def main():
    input_dir = './superfold/array_outputs'
    output_file = './superfold/array_outputs/sequences.fasta'
    
    all_sequences = []
    
    for filename in os.listdir(input_dir):
        if filename.endswith('.pdb'):
            pdb_file = os.path.join(input_dir, filename)
            pdb_id = os.path.splitext(filename)[0]
            sequences = extract_sequences_from_pdb(pdb_file)
            for chain_id, sequence in sequences:
                all_sequences.append((pdb_id, chain_id, sequence))
    
    write_fasta(all_sequences, output_file)
    print(f"Sequences written to {output_file}")

if __name__ == "__main__":
    main()

Sequences written to ./superfold/array_outputs/sequences.fasta


In [17]:
import json
from Bio import SeqIO

# Define the input and output paths
input_fasta_path = './superfold/array_outputs/sequences.fasta'
output_json_path = './output_sequences.json'

# Read the sequences from the FASTA file using Biopython
def read_fasta(file_path):
    sequences = []
    for record in SeqIO.parse(file_path, "fasta"):
        sequences.append({'name': record.id, 'sequence': str(record.seq)})
    return sequences

# Determine the count based on the name
def determine_count(name):
    if "c6" in name:
        return 6
    elif "c4" in name:
        return 4
    elif "c3" in name:
        return 3
    else:
        return 1

# Generate the JSON structure
def generate_json(sequences):
    jobs = [
        {
            "name": seq['name'],
            "modelSeeds": [],
            "sequences": [
                {
                    "proteinChain": {
                        "sequence": seq['sequence'],
                        "glycans": [],
                        "modifications": [],
                        "count": determine_count(seq['name'])
                    }
                }
            ]
        } for seq in sequences
    ]
    return jobs

# Read sequences from the input FASTA file
sequences = read_fasta(input_fasta_path)

# Generate the JSON structure
json_data = generate_json(sequences)

# Write the JSON data to the output file
with open(output_json_path, 'w') as f:
    json.dump(json_data, f, indent=2)

print(f"JSON file has been created at {output_json_path}")

JSON file has been created at ./output_sequences.json


/net/software/lab/johnbercow/JohnBercow.py --verbose --order_pdbs ./ --order_name 2024_1010_array_n_cage --design_prefix LM --design_id 1 --gg_vector LM0627 --species e_coli --no_layout

In [3]:
import pandas as pd
import re

# Specify the path to your Excel file
input_file = '/home/pkwon2/arm_maker/order/2024_10_11_2024_1010_array_n_cage_e_coli_BsaI.xlsx'
output_file = '/home/pkwon2/arm_maker/order/converted_data.xlsx'

# Read the Excel file into a DataFrame
df = pd.read_excel(input_file)

# Create 'location' column by converting 'Well Position' to lowercase
df['location'] = df['Well Position'].str.lower()

# Function to extract the 'gene name' with 'Well Position' as prefix
def extract_gene_name(full_name):
    # Extract the 'Well Position' from the 'Name' column
    # It is the second element when splitting by '__'
    parts = full_name.split('__')
    well_position = parts[1] if len(parts) > 1 else ''
    
    # Remove prefix up to '2024_1010_array_n_cage.1__'
    prefix_pattern = r'^.*2024_1010_array_n_cage\.1__'
    name = re.sub(prefix_pattern, '', full_name)
    
    # Remove suffix starting with '_model_3_ptm_seed_0_unrelaxed'
    suffix_pattern = r'_model_3_ptm_seed_0_unrelaxed.*$'
    name = re.sub(suffix_pattern, '', name)
    
    # Extract the initial 'cX_YYY' pattern
    match_initial = re.search(r'(c\d+_\d+)', name)
    initial = match_initial.group(1) if match_initial else ''
    
    # Extract all '_packed_X_Y' patterns, including at the end
    packed_matches = re.findall(r'_packed_(\d+_\d+)(?:_|$)', name)
    
    # Extract the 'tile' or 'cage' pattern
    special_match = re.search(r'_(tile_[a-z]+_\d+|cage_\d+)_', name)
    special = special_match.group(1) if special_match else ''
    
    # Assemble the parts
    parts = [well_position]  # Start with the 'Well Position'
    if initial:
        parts.append(initial)
    if packed_matches:
        parts.append(packed_matches[0])  # First '_packed_' value
    if special:
        parts.append(special)
    if len(packed_matches) > 1:
        parts.append(packed_matches[1])  # Second '_packed_' value
    
    gene_name = '_'.join(parts)
    return gene_name

# Apply the function to the 'Name' column to get the 'gene name'
df['gene name'] = df['Name'].apply(extract_gene_name)

# Rename 'Sequence' to 'gene sequence'
df['gene sequence'] = df['Sequence']

# Add 'plate' column: 1 for the first 96 rows, 2 for the next 96, etc.
df['plate'] = (df.index // 96) + 1

# Select the desired columns in the specified order
df_result = df[['gene name', 'gene sequence', 'plate', 'location']]

# Save the transformed data to a new Excel file
df_result.to_excel(output_file, index=False)

print(f"Data has been successfully converted and saved to {output_file}.")

Data has been successfully converted and saved to /home/pkwon2/arm_maker/order/converted_data.xlsx.
